In [ ]:
# Notebook 13: Demo Interface

!pip install gradio transformers torch chromadb sentence-transformers -q


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.7/20.7 MB 33.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 22.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 33.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.3/132.3 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.9/65.9 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.0/208.0 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.4/105.4 kB 5.0 MB/s eta 0:

In [ ]:
from pathlib import Path

# Google Drive mount
from google.colab import drive
drive.mount('/content/drive')

# Base directory - all data saved here
BASE_DIR = Path("/content/drive/MyDrive/multi_agent_rag_project")
BASE_DIR.mkdir(parents=True, exist_ok=True)

# Now redefine all paths
DATA_DIR = BASE_DIR / "data/raw/arxiv_papers"
CHUNKS_DIR = BASE_DIR / "data/processed/chunks"
CHROMA_DIR = BASE_DIR / "data/vector_store"
MODEL_DIR = BASE_DIR / "models"
RESULTS_DIR = BASE_DIR / "results"

# Create all directories
for dir_path in [DATA_DIR, CHUNKS_DIR, CHROMA_DIR, MODEL_DIR, RESULTS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:

import gradio as gr
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from pathlib import Path
import chromadb
from sentence_transformers import SentenceTransformer
import json

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Using device: cpu


In [ ]:


# Load models
print("Loading models for demo...")

MODEL_DIR = BASE_DIR / "models/sft_model/final"  # Changed from ppo
CHROMA_DIR = BASE_DIR / "data/vector_store"

# LLM
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
).to(DEVICE)
model.eval()

# Embedding model
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# Vector store
chroma_client = chromadb.PersistentClient(path=str(CHROMA_DIR))
collection = chroma_client.get_collection("arxiv_papers")

print("Models loaded")

Loading models for demo...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Models loaded


In [ ]:

# RAG pipeline
def retrieve_and_generate(query, use_rag=True, num_sources=5, temperature=0.7):
    """Complete RAG pipeline"""

    sources_text = ""

    if use_rag:
        # Retrieve
        query_embedding = embedding_model.encode([query])[0].tolist()
        results = collection.query(
            query_embeddings=[query_embedding],
            n_results=num_sources
        )

        # Format context
        contexts = []
        for i, (doc, metadata) in enumerate(zip(
            results['documents'][0],
            results['metadatas'][0]
        ), 1):
            contexts.append(f"[{i}] {metadata['title']}\n{doc[:300]}...")
            sources_text += f"\n**Source {i}:** {metadata['title']} (arXiv:{metadata['arxiv_id']})\n"

        context = "\n\n".join(contexts)

        prompt = f"""Context from research papers:
{context}

Question: {query}

Answer:"""
    else:
        prompt = query

    # Generate
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512).to(DEVICE)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            temperature=temperature,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id,
            no_repeat_ngram_size=3
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Clean response
    if "Answer:" in response:
        response = response.split("Answer:")[-1].strip()
    elif query in response:
        response = response.replace(query, "").strip()

    return response, sources_text if use_rag else "RAG disabled - no sources used"

In [ ]:

# Gradio interface
demo = gr.Blocks(title="Multi-Agent RAG Research Assistant", theme=gr.themes.Soft())

with demo:
    gr.Markdown("""
    #  Multi-Agent RAG Research Assistant
    ### RLHF-Aligned AI for Academic Research

    This system combines:
    -  **10K+ ArXiv papers** in Machine Learning, AI, NLP, and Computer Vision
    -  **Semantic search** with RAG (Retrieval Augmented Generation)
    -  **SFT tuned ** for high-quality, well-cited responses


    Ask questions about research papers, ML concepts, or recent advances!
    """)

    with gr.Row():
        with gr.Column(scale=2):
            query_input = gr.Textbox(
                label="Your Question",
                placeholder="E.g., What are transformers in deep learning?",
                lines=3
            )

            with gr.Row():
                use_rag = gr.Checkbox(label="Use RAG (recommended)", value=True)
                num_sources = gr.Slider(1, 10, value=5, step=1, label="Number of sources")
                temperature = gr.Slider(0.1, 1.0, value=0.7, step=0.1, label="Temperature")

            submit_btn = gr.Button("Ask", variant="primary", size="lg")

        with gr.Column(scale=1):
            examples = gr.Examples(
                examples=[
                    "What are transformers in deep learning?",
                    "Explain RLHF and how it works",
                    "What is the attention mechanism?",
                    "How does LoRA fine-tuning work?",
                    "What are diffusion models?",
                    "Explain retrieval augmented generation"
                ],
                inputs=query_input,
                label="Example Questions"
            )

    gr.Markdown("---")

    with gr.Row():
        with gr.Column():
            response_output = gr.Textbox(
                label="Response",
                lines=10,
                show_copy_button=True
            )

        with gr.Column():
            sources_output = gr.Markdown(label="Sources")

    # Statistics
    gr.Markdown(f"""
    ###  System Statistics
    - **Documents in knowledge base:** {collection.count():,}
    - **Model:** GPT-2 Medium (355M params) with LoRA + SFT
    - **Embedding model:** all-MiniLM-L6-v2
    - **Device:** {DEVICE}
    """)

    # Connect
    submit_btn.click(
        fn=retrieve_and_generate,
        inputs=[query_input, use_rag, num_sources, temperature],
        outputs=[response_output, sources_output]
    )


In [ ]:
# Cell 6
# Launch demo
print("\nLaunching Gradio demo...")
print("The interface will open in your browser")
print("Share link will be generated for remote access")

demo.launch(
    share=True,  # Creates public URL
    server_name="0.0.0.0",
    server_port=8000,
    show_error=True
)


Launching Gradio demo...
The interface will open in your browser
Share link will be generated for remote access
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://869ffd7a8dce160087.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [ ]:

print("\n" + "="*80)
print("DEMO INTERFACE LAUNCHED")
print("="*80)
print("""
The Gradio interface is now running!

Features:
- Interactive chat interface
- RAG toggle (enable/disable retrieval)
- Adjustable number of sources
- Temperature control for generation
- Example questions
- Source citations
- Public share link (if share=True)

To stop the server, interrupt the kernel.
""")







DEMO INTERFACE LAUNCHED

The Gradio interface is now running!

Features:
- Interactive chat interface
- RAG toggle (enable/disable retrieval)
- Adjustable number of sources
- Temperature control for generation
- Example questions
- Source citations
- Public share link (if share=True)

To stop the server, interrupt the kernel.

